In [1]:

import os

print("Available Kaggle input datasets:\n")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level > 2:
        dirs[:] = []
        continue

    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")


Available Kaggle input datasets:

input/
  notebooks/
    drrajgauravmishra/
  datasets/
    drrajgauravmishra/


In [2]:

import os

search_root = "/kaggle/input"

keywords = [
    ".pt",
    ".npy",
    "spatial_encoder",
    "training_annual.npz",
    "validation_annual.npz",
    "testing_annual.npz",
    "static_spatial_graph_verified.pt"
]

print("SEARCHING FOR REQUIRED FILES:\n")

for root, dirs, files in os.walk(search_root):
    for file in files:
        if any(key in file.lower() for key in keywords):
            print(os.path.join(root, file))


SEARCHING FOR REQUIRED FILES:

/kaggle/input/notebooks/drrajgauravmishra/03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/spatial_encoder_results/GAT_seed_123_training_embeddings.npy
/kaggle/input/notebooks/drrajgauravmishra/03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/spatial_encoder_results/GAT_seed_2023_training_embeddings.npy
/kaggle/input/notebooks/drrajgauravmishra/03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/spatial_encoder_results/GAT_seed_42_training_embeddings.npy
/kaggle/input/notebooks/drrajgauravmishra/03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/spatial_encoder_results/GATv2_seed_123_training_embeddings.npy
/kaggle/input/notebooks/drrajgauravmishra/03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/spatial_encoder_results/GATv2_seed_2023_training_embeddings.npy
/kaggle/input/notebooks/drrajgauravmishra/03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/spatial_encoder_results/GATv2_seed_42_training_embeddings.npy
/kaggle/input/notebooks/drrajgauravmishra/03-

In [3]:
CHECKPOINT_DIR = (
    "/kaggle/input/notebooks/drrajgauravmishra/"
    "03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/"
    "spatial_encoder_results/checkpoints"
)

EMBEDDING_DIR = (
    "/kaggle/input/notebooks/drrajgauravmishra/"
    "03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/"
    "spatial_encoder_results"
)

DATA_DIR = (
    "/kaggle/input/datasets/drrajgauravmishra/spatiotemporal-gnn-kaggle-data/Spatiotemporal_GNN_Kaggle_Data"
)

In [4]:

import os
import numpy as np
import torch

# --------------------------------------------------
# 1. Inspect annual datasets
# --------------------------------------------------

for split in ["training", "validation", "testing"]:

    path = os.path.join(DATA_DIR, f"{split}_annual.npz")

    data = np.load(path)

    print(f"\n{'='*55}")
    print(f"{split.upper()} DATASET")

    for key in data.files:
        arr = data[key]
        print(f"{key:20s} | Shape: {arr.shape} | Dtype: {arr.dtype}")

    if "input_indices" in data.files:
        print("Input indices:")
        print(data["input_indices"])

    if "target_years" in data.files:
        print("Target years:")
        print(data["target_years"])


# --------------------------------------------------
# 2. Inspect GATv2 checkpoint structure
# --------------------------------------------------

print("\n" + "="*55)
print("GATv2 CHECKPOINT INSPECTION")

for seed in [42, 123, 2023]:

    path = os.path.join(CHECKPOINT_DIR, f"GATv2_seed_{seed}.pt")

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print(f"\nSeed: {seed}")
    print("Checkpoint type:", type(checkpoint))

    if isinstance(checkpoint, dict):
        print("Checkpoint keys:", list(checkpoint.keys()))

        if "model_state_dict" in checkpoint:
            state = checkpoint["model_state_dict"]
        elif "state_dict" in checkpoint:
            state = checkpoint["state_dict"]
        else:
            state = checkpoint

        if isinstance(state, dict):
            print("Model parameter names:")
            for key, value in state.items():
                if torch.is_tensor(value):
                    print(f"  {key}: {tuple(value.shape)}")



TRAINING DATASET
X                    | Shape: (8, 10, 5040, 4) | Dtype: float32
Y                    | Shape: (8, 5040, 2) | Dtype: float32
Y_mask               | Shape: (8, 5040, 2) | Dtype: bool
input_indices        | Shape: (8, 10) | Dtype: int64
target_indices       | Shape: (8,) | Dtype: int64
target_years         | Shape: (8,) | Dtype: int64
Input indices:
[[ 0  1  2  3  4  5  6  7  8  9]
 [ 1  2  3  4  5  6  7  8  9 10]
 [ 2  3  4  5  6  7  8  9 10 11]
 [ 3  4  5  6  7  8  9 10 11 12]
 [ 4  5  6  7  8  9 10 11 12 13]
 [ 5  6  7  8  9 10 11 12 13 14]
 [ 6  7  8  9 10 11 12 13 14 15]
 [ 7  8  9 10 11 12 13 14 15 16]]
Target years:
[2010 2011 2012 2013 2014 2015 2016 2017]

VALIDATION DATASET
X                    | Shape: (6, 10, 5040, 4) | Dtype: float32
Y                    | Shape: (6, 5040, 2) | Dtype: float32
Y_mask               | Shape: (6, 5040, 2) | Dtype: bool
input_indices        | Shape: (6, 10) | Dtype: int64
target_indices       | Shape: (6,) | Dtype: int64
target_y

In [5]:

import numpy as np
import os

# Annual indices 0 to 24 represent 2000 to 2024
NUM_YEARS = 25
NUM_NODES = 5040
NUM_FEATURES = 2

annual_snapshots = [None] * NUM_YEARS

split_files = [
    os.path.join(DATA_DIR, "training_annual.npz"),
    os.path.join(DATA_DIR, "validation_annual.npz"),
    os.path.join(DATA_DIR, "testing_annual.npz")
]

# Reconstruct annual snapshots from overlapping input windows
for file_path in split_files:

    data = np.load(file_path)

    X = data["X"]
    input_indices = data["input_indices"]

    for sample_idx in range(X.shape[0]):

        for time_idx in range(X.shape[1]):

            year_idx = int(input_indices[sample_idx, time_idx])

            if year_idx >= NUM_YEARS:
                continue

            # Retain only rainfall and temperature channels
            snapshot = X[sample_idx, time_idx, :, :2]

            if annual_snapshots[year_idx] is None:
                annual_snapshots[year_idx] = snapshot.copy()

            else:
                # Verify consistency of overlapping annual snapshots
                if not np.allclose(
                    annual_snapshots[year_idx],
                    snapshot,
                    atol=1e-6,
                    equal_nan=True
                ):
                    raise ValueError(
                        f"Inconsistent data for year index {year_idx}"
                    )

# Verify complete annual coverage
missing_years = [
    2000 + i
    for i, snapshot in enumerate(annual_snapshots)
    if snapshot is None
]

if missing_years:
    raise ValueError(f"Missing annual snapshots: {missing_years}")

annual_snapshots_np = np.stack(annual_snapshots, axis=0)

print("Annual snapshot reconstruction successful!")
print("Shape:", annual_snapshots_np.shape)
print("Years: 2000–2024")
print("Number of nodes:", annual_snapshots_np.shape[1])
print("Climate features:", annual_snapshots_np.shape[2])
print("All overlapping snapshots are consistent.")
print("NaN values:", np.isnan(annual_snapshots_np).sum())


Annual snapshot reconstruction successful!
Shape: (25, 5040, 2)
Years: 2000–2024
Number of nodes: 5040
Climate features: 2
All overlapping snapshots are consistent.
NaN values: 0


In [6]:
pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

class GATv2Encoder(nn.Module):

    def __init__(self, input_dim=2, hidden_dim=64, output_dim=2):
        super().__init__()

        self.conv1 = GATv2Conv(
            input_dim,
            hidden_dim,
            heads=4,
            concat=True,
            add_self_loops=False
        )

        self.conv2 = GATv2Conv(
            hidden_dim * 4,
            output_dim,
            heads=1,
            concat=True,
            add_self_loops=False
        )

    def forward(self, x, edge_index):

        hidden = self.conv1(x, edge_index)
        hidden = F.relu(hidden)

        reconstruction = self.conv2(hidden, edge_index)

        return reconstruction, hidden


# Initialize model and load one checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GATv2Encoder().to(device)

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "GATv2_seed_42.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Checkpoint loaded successfully.")
print("Model device:", device)
print("Hidden embedding dimension:", 256)
print("Checkpoint seed:", checkpoint["seed"])


Checkpoint loaded successfully.
Model device: cuda
Hidden embedding dimension: 256
Checkpoint seed: 42


In [8]:

import os
import numpy as np
import torch

# --------------------------------------------------
# 1. Load spatial graph
# --------------------------------------------------

graph_path = os.path.join(
    DATA_DIR,
    "static_spatial_graph_verified.pt"
)

graph_data = torch.load(
    graph_path,
    map_location="cpu",
    weights_only=False
)

edge_index = graph_data.edge_index.to(device)

print("Graph nodes:", graph_data.num_nodes)
print("Graph edges:", edge_index.shape[1])


# --------------------------------------------------
# 2. Prepare output directory
# --------------------------------------------------

OUTPUT_DIR = "/kaggle/working/gatv2_complete_embeddings"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# --------------------------------------------------
# 3. Extract embeddings for all seeds and years
# --------------------------------------------------

seeds = [42, 123, 2023]

embedding_results = {}

for seed in seeds:

    print(f"\nProcessing GATv2 | Seed {seed}")

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"GATv2_seed_{seed}.pt"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model = GATv2Encoder().to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    yearly_embeddings = []

    with torch.no_grad():

        for year_idx in range(25):

            x = torch.tensor(
                annual_snapshots_np[year_idx],
                dtype=torch.float32,
                device=device
            )

            reconstruction, hidden = model(x, edge_index)

            embedding = hidden.cpu().numpy()

            yearly_embeddings.append(embedding)

            if (year_idx + 1) % 5 == 0:
                print(
                    f"Completed year: {2000 + year_idx}"
                )

    yearly_embeddings = np.stack(
        yearly_embeddings,
        axis=0
    )

    embedding_results[seed] = yearly_embeddings

    output_path = os.path.join(
        OUTPUT_DIR,
        f"GATv2_seed_{seed}_embeddings_2000_2024.npy"
    )

    np.save(output_path, yearly_embeddings)

    print("Saved:", output_path)
    print("Embedding shape:", yearly_embeddings.shape)


print("\nAll three GATv2 embedding extractions completed.")


Graph nodes: 5040
Graph edges: 40320

Processing GATv2 | Seed 42
Completed year: 2004
Completed year: 2009
Completed year: 2014
Completed year: 2019
Completed year: 2024
Saved: /kaggle/working/gatv2_complete_embeddings/GATv2_seed_42_embeddings_2000_2024.npy
Embedding shape: (25, 5040, 256)

Processing GATv2 | Seed 123
Completed year: 2004
Completed year: 2009
Completed year: 2014
Completed year: 2019
Completed year: 2024
Saved: /kaggle/working/gatv2_complete_embeddings/GATv2_seed_123_embeddings_2000_2024.npy
Embedding shape: (25, 5040, 256)

Processing GATv2 | Seed 2023
Completed year: 2004
Completed year: 2009
Completed year: 2014
Completed year: 2019
Completed year: 2024
Saved: /kaggle/working/gatv2_complete_embeddings/GATv2_seed_2023_embeddings_2000_2024.npy
Embedding shape: (25, 5040, 256)

All three GATv2 embedding extractions completed.


In [9]:

import os
import numpy as np

print("VERIFYING TRAINING EMBEDDING CONSISTENCY\n")

for seed in [42, 123, 2023]:

    old_path = os.path.join(
        EMBEDDING_DIR,
        f"GATv2_seed_{seed}_training_embeddings.npy"
    )

    new_path = os.path.join(
        OUTPUT_DIR,
        f"GATv2_seed_{seed}_embeddings_2000_2024.npy"
    )

    old_embeddings = np.load(old_path)
    new_embeddings = np.load(new_path)

    # Compare 2000–2016
    difference = np.abs(
        old_embeddings - new_embeddings[:17]
    )

    max_difference = difference.max()
    mean_difference = difference.mean()

    print(f"Seed: {seed}")
    print("Original shape:", old_embeddings.shape)
    print("New shape:", new_embeddings.shape)
    print("Maximum absolute difference:", max_difference)
    print("Mean absolute difference:", mean_difference)

    print("Numerically consistent:",
          np.allclose(
              old_embeddings,
              new_embeddings[:17],
              atol=1e-5,
              rtol=1e-5
          ))
    print("-" * 55)


VERIFYING TRAINING EMBEDDING CONSISTENCY

Seed: 42
Original shape: (17, 5040, 256)
New shape: (25, 5040, 256)
Maximum absolute difference: 4.7683716e-07
Mean absolute difference: 4.7697557e-09
Numerically consistent: True
-------------------------------------------------------
Seed: 123
Original shape: (17, 5040, 256)
New shape: (25, 5040, 256)
Maximum absolute difference: 9.536743e-07
Mean absolute difference: 5.268411e-09
Numerically consistent: True
-------------------------------------------------------
Seed: 2023
Original shape: (17, 5040, 256)
New shape: (25, 5040, 256)
Maximum absolute difference: 1.4305115e-06
Mean absolute difference: 5.696765e-09
Numerically consistent: True
-------------------------------------------------------
